<a href="https://www.kaggle.com/code/kedhareswernaidu/moonknight?scriptVersionId=228859653" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

## Like Old Paintings

In [ ]:
import os
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import random
from typing import Tuple
import torch.nn as nn
import torchvision.models as models
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np

# Define the custom dataset class
class ImageFolderCustom(Dataset):
    def __init__(self, root: str, class_idx: int, max_size: int = -1, transform=None):
        self.paths = [os.path.join(root, img) for img in os.listdir(root)]
        random.shuffle(self.paths)
        if max_size != -1:
            self.paths = self.paths[:max_size]
        self.transform = transform
        self.class_idx = class_idx

    def load_image(self, index: int) -> Image.Image:
        image_path = self.paths[index]
        return Image.open(image_path)

    def __len__(self) -> int:
        return len(self.paths)

    def __getitem__(self, index: int) -> Tuple[torch.Tensor, int]:
        img = self.load_image(index)
        if self.transform:
            return self.transform(img), self.class_idx
        else:
            return img, self.class_idx

# Define transformations
data_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load datasets
monet_dataset = ImageFolderCustom(root="/kaggle/input/c/gan-getting-started/monet_jpg",
                                  class_idx=1,
                                  max_size=32,
                                  transform=data_transform)
photo_dataset = ImageFolderCustom(root="/kaggle/input/c/gan-getting-started/photo_jpg",
                                  class_idx=0,
                                  transform=data_transform)

# Create data loaders
monet_dl = DataLoader(dataset=monet_dataset, batch_size=1, shuffle=True)
photo_dl = DataLoader(dataset=photo_dataset, batch_size=1, shuffle=True)

# Define the VGG model
class VGG(nn.Module):
    def __init__(self):
        super(VGG, self).__init__()
        self.select_features = ['0', '5', '10', '19', '28']
        self.vgg = models.vgg19(weights=models.VGG19_Weights.IMAGENET1K_V1)
        self.vgg.features = nn.Sequential(*list(self.vgg.features.children())[:29])

    def forward(self, x):
        features = []
        for name, layer in self.vgg.features._modules.items():
            x = layer(x)
            if name in self.select_features:
                features.append(x)
        return features

# Define the function to get features
def get_features(image, model, layers=None):
    if layers is None:
        layers = {'0': 'conv1_1',
                  '5': 'conv2_1',
                  '10': 'conv3_1',
                  '19': 'conv4_1',
                  '28': 'conv5_1'}
    features = {}
    x = image
    for name, layer in model.vgg.features._modules.items():
        x = layer(x)
        if name in layers:
            features[layers[name]] = x
    return features

# Define the Gram matrix function
def gram_matrix(tensor):
    _, d, h, w = tensor.size()
    tensor = tensor.view(d, h * w)
    gram = torch.mm(tensor, tensor.t())
    return gram

# Define the load_image function
def load_image(image_path, transform=None, max_size=None, shape=None):
    image = Image.open(image_path).convert('RGB')
    if max_size is not None:
        image = transforms.functional.resize(image, max_size)
    if shape is not None:
        image = transforms.functional.resize(image, shape)
    if transform is not None:
        image = transform(image)
    return image.unsqueeze(0)

# Set up the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Randomly select a content image from the photo dataset
content_image_path = random.choice(photo_dataset.paths)
content_image = load_image(content_image_path, transform=data_transform).to(device)

# Randomly select a style image from the monet dataset
style_image_path = random.choice(monet_dataset.paths)
style_image = load_image(style_image_path, transform=data_transform).to(device)

# Initialize the generated image with the content image
generated_image = content_image.clone().requires_grad_(True)

# Define the optimizer
optimizer = optim.Adam([generated_image], lr=0.003)

# Define the VGG model
vgg = VGG().to(device).eval()

# Training loop
num_epochs = 3000
content_weight = 1e5
style_weight = 1e10
tv_weight = 1e-6  # Total variation loss weight

for epoch in range(num_epochs):
    generated_features = get_features(generated_image, vgg)
    content_features = get_features(content_image, vgg)
    style_features = get_features(style_image, vgg)

    # Compute content loss
    content_loss = torch.mean((generated_features['conv4_1'] - content_features['conv4_1']) ** 2)

    # Compute style loss
    style_loss = 0
    for layer in style_features.keys():
        generated_gram = gram_matrix(generated_features[layer])
        style_gram = gram_matrix(style_features[layer])
        style_loss += torch.mean((generated_gram - style_gram) ** 2)

    # Compute total variation loss
    tv_loss = torch.sum(torch.abs(generated_image[:, :, :, :-1] - generated_image[:, :, :, 1:])) + \
              torch.sum(torch.abs(generated_image[:, :, :-1, :] - generated_image[:, :, 1:, :]))

    # Total loss
    total_loss = content_weight * content_loss + style_weight * style_loss + tv_weight * tv_loss

    # Backpropagation
    optimizer.zero_grad()
    total_loss.backward()
    optimizer.step()

    if epoch % 100 == 0:
        print(f"Epoch {epoch}, Total Loss: {total_loss.item()}")

# Save the generated image
torch.save(generated_image, 'generated_image.pth')

# Display the original and generated images
def im_convert(tensor):
    image = tensor.to("cpu").clone().detach()
    image = image.numpy().squeeze()
    image = image.transpose(1, 2, 0)
    image = image * np.array((0.229, 0.224, 0.225)) + np.array((0.485, 0.456, 0.406))
    image = image.clip(0, 1)
    return image

fig, ax = plt.subplots(1, 2, figsize=(10, 5))

# Original content image
ax[0].imshow(im_convert(content_image))
ax[0].set_title('Original Content Image')
ax[0].axis('off')

# Generated image
ax[1].imshow(im_convert(generated_image))
ax[1].set_title('Generated Image')
ax[1].axis('off')

plt.show()

In [ ]:
print(generated_features.keys())
print(content_features.keys())

In [ ]:
import os
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import random
from typing import Tuple
import torch.nn as nn
import torchvision.models as models
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np

# Define the custom dataset class
class ImageFolderCustom(Dataset):
    def __init__(self, root: str, class_idx: int, max_size: int = -1, transform=None):
        self.paths = [os.path.join(root, img) for img in os.listdir(root)]
        random.shuffle(self.paths)
        if max_size != -1:
            self.paths = self.paths[:max_size]
        self.transform = transform
        self.class_idx = class_idx

    def load_image(self, index: int) -> Image.Image:
        image_path = self.paths[index]
        return Image.open(image_path)

    def __len__(self) -> int:
        return len(self.paths)

    def __getitem__(self, index: int) -> Tuple[torch.Tensor, int]:
        img = self.load_image(index)
        if self.transform:
            return self.transform(img), self.class_idx
        else:
            return img, self.class_idx

# Define transformations
data_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load datasets
monet_dataset = ImageFolderCustom(root="/kaggle/input/c/gan-getting-started/monet_jpg",
                                  class_idx=1,
                                  max_size=32,
                                  transform=data_transform)
photo_dataset = ImageFolderCustom(root="/kaggle/input/c/gan-getting-started/photo_jpg",
                                  class_idx=0,
                                  transform=data_transform)

# Create data loaders
monet_dl = DataLoader(dataset=monet_dataset, batch_size=1, shuffle=True)
photo_dl = DataLoader(dataset=photo_dataset, batch_size=1, shuffle=True)

# Define the VGG model
class VGG(nn.Module):
    def __init__(self):
        super(VGG, self).__init__()
        self.select_features = ['0', '5', '10', '19', '28']
        self.vgg = models.vgg19(weights=models.VGG19_Weights.IMAGENET1K_V1)
        self.vgg.features = nn.Sequential(*list(self.vgg.features.children())[:29])

    def forward(self, x):
        features = []
        for name, layer in self.vgg.features._modules.items():
            x = layer(x)
            if name in self.select_features:
                features.append(x)
        return features

# Define the function to get features
def get_features(image, model, layers=None):
    if layers is None:
        layers = {'0': 'conv1_1',
                  '5': 'conv2_1',
                  '10': 'conv3_1',
                  '19': 'conv4_1',
                  '28': 'conv5_1'}
    features = {}
    x = image
    for name, layer in model.vgg.features._modules.items():
        x = layer(x)
        if name in layers:
            features[layers[name]] = x
    return features

# Define the Gram matrix function
def gram_matrix(tensor):
    _, d, h, w = tensor.size()
    tensor = tensor.view(d, h * w)
    gram = torch.mm(tensor, tensor.t())
    return gram

# Define the load_image function
def load_image(image_path, transform=None, max_size=None, shape=None):
    image = Image.open(image_path).convert('RGB')
    if max_size is not None:
        image = transforms.functional.resize(image, max_size)
    if shape is not None:
        image = transforms.functional.resize(image, shape)
    if transform is not None:
        image = transform(image)
    return image.unsqueeze(0)

# Set up the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Randomly select a content image from the photo dataset
content_image_path = random.choice(photo_dataset.paths)
content_image = load_image(content_image_path, transform=data_transform).to(device)

# Randomly select a style image from the monet dataset
style_image_path = random.choice(monet_dataset.paths)
style_image = load_image(style_image_path, transform=data_transform).to(device)

# Initialize the generated image with the content image
generated_image = content_image.clone().requires_grad_(True)

# Define the optimizer
optimizer = optim.Adam([generated_image], lr=0.003)

# Define the VGG model
vgg = VGG().to(device).eval()

# Training loop
num_epochs = 3000
content_weight = 1e5
style_weight = 1e10
tv_weight = 1e-6  # Total variation loss weight

for epoch in range(num_epochs):
    generated_features = get_features(generated_image, vgg)
    content_features = get_features(content_image, vgg)
    style_features = get_features(style_image, vgg)

    # Compute content loss
    content_loss = torch.mean((generated_features['conv4_1'] - content_features['conv4_1']) ** 2)

    # Compute style loss
    style_loss = 0
    for layer in style_features.keys():
        generated_gram = gram_matrix(generated_features[layer])
        style_gram = gram_matrix(style_features[layer])
        style_loss += torch.mean((generated_gram - style_gram) ** 2)

    # Compute total variation loss
    tv_loss = torch.sum(torch.abs(generated_image[:, :, :, :-1] - generated_image[:, :, :, 1:])) + \
              torch.sum(torch.abs(generated_image[:, :, :-1, :] - generated_image[:, :, 1:, :]))

    # Total loss
    total_loss = content_weight * content_loss + style_weight * style_loss + tv_weight * tv_loss

    # Backpropagation
    optimizer.zero_grad()
    total_loss.backward()
    optimizer.step()

    if epoch % 100 == 0:
        print(f"Epoch {epoch}, Total Loss: {total_loss.item()}")

# Save the generated image
torch.save(generated_image, 'generated_image.pth')

# Display the original and generated images
def im_convert(tensor):
    image = tensor.to("cpu").clone().detach()
    image = image.numpy().squeeze()
    image = image.transpose(1, 2, 0)
    image = image * np.array((0.229, 0.224, 0.225)) + np.array((0.485, 0.456, 0.406))
    image = image.clip(0, 1)
    return image

fig, ax = plt.subplots(1, 2, figsize=(10, 5))

# Original content image
ax[0].imshow(im_convert(content_image))
ax[0].set_title('Original Content Image')
ax[0].axis('off')

# Generated image
ax[1].imshow(im_convert(generated_image))
ax[1].set_title('Generated Image')
ax[1].axis('off')

plt.show()

In [ ]:
import os
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import random
from typing import Tuple
import torch.nn as nn
import torchvision.models as models
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np

# Define the custom dataset class
class ImageFolderCustom(Dataset):
    def __init__(self, root: str, class_idx: int, max_size: int = -1, transform=None):
        self.paths = [os.path.join(root, img) for img in os.listdir(root)]
        random.shuffle(self.paths)
        if max_size != -1:
            self.paths = self.paths[:max_size]
        self.transform = transform
        self.class_idx = class_idx

    def load_image(self, index: int) -> Image.Image:
        image_path = self.paths[index]
        return Image.open(image_path)

    def __len__(self) -> int:
        return len(self.paths)

    def __getitem__(self, index: int) -> Tuple[torch.Tensor, int]:
        img = self.load_image(index)
        if self.transform:
            return self.transform(img), self.class_idx
        else:
            return img, self.class_idx

# Define transformations
data_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load datasets
monet_dataset = ImageFolderCustom(root="/kaggle/input/c/gan-getting-started/monet_jpg",
                                  class_idx=1,
                                  max_size=32,
                                  transform=data_transform)
photo_dataset = ImageFolderCustom(root="/kaggle/input/c/gan-getting-started/photo_jpg",
                                  class_idx=0,
                                  transform=data_transform)

# Create data loaders
monet_dl = DataLoader(dataset=monet_dataset, batch_size=1, shuffle=True)
photo_dl = DataLoader(dataset=photo_dataset, batch_size=1, shuffle=True)

# Define the VGG model
class VGG(nn.Module):
    def __init__(self):
        super(VGG, self).__init__()
        self.select_features = ['0', '5', '10', '19', '28']
        self.vgg = models.vgg19(weights=models.VGG19_Weights.IMAGENET1K_V1)
        self.vgg.features = nn.Sequential(*list(self.vgg.features.children())[:29])

    def forward(self, x):
        features = []
        for name, layer in self.vgg.features._modules.items():
            x = layer(x)
            if name in self.select_features:
                features.append(x)
        return features

# Define the function to get features
def get_features(image, model, layers=None):
    if layers is None:
        layers = {'0': 'conv1_1',
                  '5': 'conv2_1',
                  '10': 'conv3_1',
                  '19': 'conv4_1',
                  '28': 'conv5_1'}
    features = {}
    x = image
    for name, layer in model.vgg.features._modules.items():
        x = layer(x)
        if name in layers:
            features[layers[name]] = x
    return features

# Define the Gram matrix function
def gram_matrix(tensor):
    _, d, h, w = tensor.size()
    tensor = tensor.view(d, h * w)
    gram = torch.mm(tensor, tensor.t())
    return gram

# Define the load_image function
def load_image(image_path, transform=None, max_size=None, shape=None):
    image = Image.open(image_path).convert('RGB')
    if max_size is not None:
        image = transforms.functional.resize(image, max_size)
    if shape is not None:
        image = transforms.functional.resize(image, shape)
    if transform is not None:
        image = transform(image)
    return image.unsqueeze(0)

# Set up the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Function to train and generate an image
def train_and_generate(content_image_path, style_image_path):
    content_image = load_image(content_image_path, transform=data_transform).to(device)
    style_image = load_image(style_image_path, transform=data_transform).to(device)

    generated_image = content_image.clone().requires_grad_(True)
    optimizer = optim.Adam([generated_image], lr=0.003)
    vgg = VGG().to(device).eval()

    num_epochs = 3000
    content_weight = 1e5
    style_weight = 1e10
    tv_weight = 1e-6

    for epoch in range(num_epochs):
        generated_features = get_features(generated_image, vgg)
        content_features = get_features(content_image, vgg)
        style_features = get_features(style_image, vgg)

        content_loss = torch.mean((generated_features['conv4_1'] - content_features['conv4_1']) ** 2)
        style_loss = 0
        for layer in style_features.keys():
            generated_gram = gram_matrix(generated_features[layer])
            style_gram = gram_matrix(style_features[layer])
            style_loss += torch.mean((generated_gram - style_gram) ** 2)

        tv_loss = torch.sum(torch.abs(generated_image[:, :, :, :-1] - generated_image[:, :, :, 1:])) + \
                  torch.sum(torch.abs(generated_image[:, :, :-1, :] - generated_image[:, :, 1:, :]))

        total_loss = content_weight * content_loss + style_weight * style_loss + tv_weight * tv_loss

        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()

        if epoch % 100 == 0:
            print(f"Epoch {epoch}, Total Loss: {total_loss.item()}")

    return generated_image

# Display the original and generated images
def im_convert(tensor):
    image = tensor.to("cpu").clone().detach()
    image = image.numpy().squeeze()
    image = image.transpose(1, 2, 0)
    image = image * np.array((0.229, 0.224, 0.225)) + np.array((0.485, 0.456, 0.406))
    image = image.clip(0, 1)
    return image

# Generate and display multiple samples
num_samples = 5
fig, axes = plt.subplots(num_samples, 2, figsize=(10, 5 * num_samples))

for i in range(num_samples):
    content_image_path = random.choice(photo_dataset.paths)
    style_image_path = random.choice(monet_dataset.paths)

    generated_image = train_and_generate(content_image_path, style_image_path)

    content_image = load_image(content_image_path, transform=data_transform)
    axes[i, 0].imshow(im_convert(content_image))
    axes[i, 0].set_title('Original Content Image')
    axes[i, 0].axis('off')

    axes[i, 1].imshow(im_convert(generated_image))
    axes[i, 1].set_title('Generated Image')
    axes[i, 1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
import os
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import random
from typing import Tuple
import torch.nn as nn
import torchvision.models as models
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np

# Define the custom dataset class
class ImageFolderCustom(Dataset):
    def __init__(self, root: str, class_idx: int, max_size: int = -1, transform=None):
        self.paths = [os.path.join(root, img) for img in os.listdir(root)]
        random.shuffle(self.paths)
        if max_size != -1:
            self.paths = self.paths[:max_size]
        self.transform = transform
        self.class_idx = class_idx

    def load_image(self, index: int) -> Image.Image:
        image_path = self.paths[index]
        return Image.open(image_path)

    def __len__(self) -> int:
        return len(self.paths)

    def __getitem__(self, index: int) -> Tuple[torch.Tensor, int]:
        img = self.load_image(index)
        if self.transform:
            return self.transform(img), self.class_idx
        else:
            return img, self.class_idx

# Define transformations
data_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load datasets
monet_dataset = ImageFolderCustom(root="/kaggle/input/c/gan-getting-started/monet_jpg",
                                  class_idx=1,
                                  max_size=32,
                                  transform=data_transform)
photo_dataset = ImageFolderCustom(root="/kaggle/input/c/gan-getting-started/photo_jpg",
                                  class_idx=0,
                                  transform=data_transform)

# Create data loaders
monet_dl = DataLoader(dataset=monet_dataset, batch_size=1, shuffle=True)
photo_dl = DataLoader(dataset=photo_dataset, batch_size=1, shuffle=True)

# Define the VGG model
class VGG(nn.Module):
    def __init__(self):
        super(VGG, self).__init__()
        self.select_features = ['0', '5', '10', '19', '28']
        self.vgg = models.vgg19(weights=models.VGG19_Weights.IMAGENET1K_V1)
        self.vgg.features = nn.Sequential(*list(self.vgg.features.children())[:29])

    def forward(self, x):
        features = []
        for name, layer in self.vgg.features._modules.items():
            x = layer(x)
            if name in self.select_features:
                features.append(x)
        return features

# Define the function to get features
def get_features(image, model, layers=None):
    if layers is None:
        layers = {'0': 'conv1_1',
                  '5': 'conv2_1',
                  '10': 'conv3_1',
                  '19': 'conv4_1',
                  '28': 'conv5_1'}
    features = {}
    x = image
    for name, layer in model.vgg.features._modules.items():
        x = layer(x)
        if name in layers:
            features[layers[name]] = x
    return features

# Define the Gram matrix function
def gram_matrix(tensor):
    _, d, h, w = tensor.size()
    tensor = tensor.view(d, h * w)
    gram = torch.mm(tensor, tensor.t())
    return gram

# Define the load_image function
def load_image(image_path, transform=None, max_size=None, shape=None):
    image = Image.open(image_path).convert('RGB')
    if max_size is not None:
        image = transforms.functional.resize(image, max_size)
    if shape is not None:
        image = transforms.functional.resize(image, shape)
    if transform is not None:
        image = transform(image)
    return image.unsqueeze(0)

# Set up the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Function to train and generate an image
def train_and_generate(content_image_path, style_image_path):
    content_image = load_image(content_image_path, transform=data_transform).to(device)
    style_image = load_image(style_image_path, transform=data_transform).to(device)

    generated_image = content_image.clone().requires_grad_(True)
    optimizer = optim.Adam([generated_image], lr=0.003)
    vgg = VGG().to(device).eval()

    num_epochs = 3000
    content_weight = 1e5
    style_weight = 1e10
    tv_weight = 1e-6

    for epoch in range(num_epochs):
        generated_features = get_features(generated_image, vgg)
        content_features = get_features(content_image, vgg)
        style_features = get_features(style_image, vgg)

        content_loss = torch.mean((generated_features['conv4_1'] - content_features['conv4_1']) ** 2)
        style_loss = 0
        for layer in style_features.keys():
            generated_gram = gram_matrix(generated_features[layer])
            style_gram = gram_matrix(style_features[layer])
            style_loss += torch.mean((generated_gram - style_gram) ** 2)

        tv_loss = torch.sum(torch.abs(generated_image[:, :, :, :-1] - generated_image[:, :, :, 1:])) + \
                  torch.sum(torch.abs(generated_image[:, :, :-1, :] - generated_image[:, :, 1:, :]))

        total_loss = content_weight * content_loss + style_weight * style_loss + tv_weight * tv_loss

        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()

        if epoch % 100 == 0:
            print(f"Epoch {epoch}, Total Loss: {total_loss.item()}")

    return generated_image

# Display the original and generated images
def im_convert(tensor):
    image = tensor.to("cpu").clone().detach()
    image = image.numpy().squeeze()
    image = image.transpose(1, 2, 0)
    image = image * np.array((0.229, 0.224, 0.225)) + np.array((0.485, 0.456, 0.406))
    image = image.clip(0, 1)
    return image

# Generate and display multiple samples
num_samples = 5
fig, axes = plt.subplots(num_samples, 2, figsize=(10, 5 * num_samples))

for i in range(num_samples):
    content_image_path = random.choice(photo_dataset.paths)
    style_image_path = random.choice(monet_dataset.paths)

    generated_image = train_and_generate(content_image_path, style_image_path)

    content_image = load_image(content_image_path, transform=data_transform)
    axes[i, 0].imshow(im_convert(content_image))
    axes[i, 0].set_title('Original Content Image')
    axes[i, 0].axis('off')

    axes[i, 1].imshow(im_convert(generated_image))
    axes[i, 1].set_title('Generated Image')
    axes[i, 1].axis('off')

plt.tight_layout()
plt.show()

Downloading: "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth" to /root/.cache/torch/hub/checkpoints/vgg19-dcbb9e9d.pth
100%|██████████| 548M/548M [00:02<00:00, 203MB/s]  


Epoch 0, Total Loss: 1.3517203537343283e+18
Epoch 100, Total Loss: 5.3142977255152026e+17
Epoch 200, Total Loss: 3.99812526990164e+17
Epoch 300, Total Loss: 3.231105958365102e+17
Epoch 400, Total Loss: 2.718588541725573e+17
Epoch 500, Total Loss: 2.3554499158422323e+17
Epoch 600, Total Loss: 2.0879227595259904e+17
Epoch 700, Total Loss: 1.8838968664798003e+17
Epoch 800, Total Loss: 1.723698194111529e+17
Epoch 900, Total Loss: 1.5943033707875533e+17
Epoch 1000, Total Loss: 1.4865842166143386e+17
Epoch 1100, Total Loss: 1.3941028333132186e+17
Epoch 1200, Total Loss: 1.3123155749817549e+17
Epoch 1300, Total Loss: 1.2384096675377971e+17
Epoch 1400, Total Loss: 1.1698796847588966e+17
Epoch 1500, Total Loss: 1.1055262999773184e+17
Epoch 1600, Total Loss: 1.0444613139562496e+17
Epoch 1700, Total Loss: 9.858824268047974e+16
Epoch 1800, Total Loss: 9.297662739008717e+16
Epoch 1900, Total Loss: 8.756459064000512e+16
Epoch 2000, Total Loss: 8.232554658267136e+16
Epoch 2100, Total Loss: 7.72556898

In [ ]:
import PIL
! mkdir ../images

In [ ]:
i = 1
for img in photo_ds:
    prediction = monet_generator(img, training=False)[0].numpy()
    prediction = (prediction * 127.5 + 127.5).astype(np.uint8)
    im = PIL.Image.fromarray(prediction)
    im.save("../images/" + str(i) + ".jpg")
    i += 1

In [ ]:
import shutil
shutil.make_archive("/kaggle/working/images", 'zip', "/kaggle/images")